# 03 — Tính frame index từ thời gian của video (AIC 2026)

## Notebook này làm gì?

Cho phép bạn **nhập mốc thời gian bất kỳ** trong một video và biết ngay **frame index** tương ứng,
kèm **ảnh cắt trực tiếp từ file mp4** tại đúng thời điểm đó.

Công thức: `frame_idx = round(số_giây × fps)`

## Vì sao cần?

`map-keyframes` chỉ chứa **một số** frame đã được chọn sẵn (mỗi video vài trăm keyframe).
Khi xem video và thấy khoảnh khắc cần tìm ở phút `10:11`, khoảnh khắc đó thường **không trùng**
với keyframe nào cả. Notebook này gỡ ràng buộc đó: bạn nhập thời gian tự do,
frame index được tính trực tiếp từ `fps` — dùng cho việc điền đáp án submit
hoặc đối chiếu, kiểm tra kết quả retrieval.

## Có gì trong đây?

| Phần | Nội dung |
|---|---|
| 1 | Cấu hình đường dẫn `DATASET` và thư mục `map-keyframes` (nguồn `fps`) |
| 2 | Hàm tiện ích: đọc thời gian `mm:ss` / `hh:mm:ss` / số giây, ghép path, lấy `fps` (có cache) |
| 3 | `time_to_frame()` và `frame_to_time()` |
| 4 | ⭐ **Ô nhập nhiều video / nhiều mốc thời gian cùng lúc** |
| 5 | Chạy → bảng kết quả + **lưới ảnh cắt từ video**; lưu ảnh ra file; xuất CSV |
| 6 | Chiều ngược lại: từ frame index → thời gian + ảnh |

## Đầu ra

Mỗi truy vấn trả về:

- `frame_idx` — frame thật trong video (đây là giá trị chính)
- `so_giay`, `fps`, `nguon_fps` — nguồn fps là `map-keyframes` hay đọc từ video
- `kich_thuoc` — kích thước ảnh cắt được
- `kf_n`, `kf_frame_idx`, `kf_lech_giay`, `kf_anh` — **keyframe gần nhất** trong `map-keyframes`,
  để bạn lấy ảnh/embedding có sẵn nếu cần, kèm độ lệch bao nhiêu giây

## Cần gì để chạy?

Trên Kaggle, thêm 2 dataset làm Input:

- `fatle542/aic-dataset` — chứa file video `.mp4`
- `kitnehi1211/feature-aic-2026` — chứa `map-keyframes` (cột `fps`)

Không cần GPU. Nếu chỉ cần số liệu mà không cần ảnh, đặt `CAT_ANH = False` ở Phần 5
thì notebook **không mở video** chút nào (chỉ đọc `fps` từ CSV) — chạy gần như tức thì.

## Cách nhập

Đường dẫn chỉ cần phần **sau** `DATASET`:

```
Videos_L26_e/video/L26_V400.mp4   10:11
```

---


## 1. Cấu hình đường dẫn

`MAP_KEYFRAMES_DIRS` là thư mục chứa các file `<video_id>.csv` (có cột `fps`).
Được lập chỉ mục bằng `os.listdir` một lần duy nhất — không quét đệ quy nên rất nhanh.

In [ ]:
import os, re
import pandas as pd

DATASET = "/kaggle/input/datasets/fatle542/aic-dataset"

# Thư mục chứa các file fps (map-keyframes). Thêm dòng mới nếu có batch khác.
MAP_KEYFRAMES_DIRS = [
    "/kaggle/input/datasets/kitnehi1211/feature-aic-2026/Feature_Dataset/map-keyframes-aic25-b1/map-keyframes",
    # "/kaggle/input/datasets/kitnehi1211/feature-aic-2026/Feature_Dataset/map-keyframes-aic25-b2/map-keyframes",
]

# Kaggle có thể mount ở dạng ngắn hơn -> tự thử cả 2 kiểu
def _chuan_hoa(p):
    if os.path.isdir(p):
        return p
    alt = p.replace("/kaggle/input/datasets/", "/kaggle/input/")
    return alt if os.path.isdir(alt) else p

DATASET = _chuan_hoa(DATASET)
MAP_KEYFRAMES_DIRS = [_chuan_hoa(d) for d in MAP_KEYFRAMES_DIRS]

print("DATASET:", DATASET, os.path.isdir(DATASET))

# Lập chỉ mục {video_id: csv_path} bằng os.listdir (không quét đệ quy -> tức thì)
MAP_KEYFRAMES = {}
for d in MAP_KEYFRAMES_DIRS:
    if not os.path.isdir(d):
        print("  [!] Không thấy:", d)
        continue
    ten = [f for f in os.listdir(d) if f.endswith(".csv")]
    for f in ten:
        MAP_KEYFRAMES[f[:-4]] = os.path.join(d, f)
    print(f"  {len(ten):5d} csv <- {d}")

print(f"Tổng: {len(MAP_KEYFRAMES)} video có fps")


## 2. Các hàm tiện ích

In [ ]:
def doc_thoi_gian(t):
    """"5:01" -> 301.0 | "1:02:03" -> 3723.0 | 301 -> 301.0"""
    if isinstance(t, (int, float)):
        return float(t)
    giay = 0.0
    for p in str(t).strip().split(":"):
        giay = giay * 60 + float(p)
    return giay


def dinh_dang_thoi_gian(giay):
    h = int(giay // 3600); m = int((giay % 3600) // 60); s = giay - h*3600 - m*60
    return f"{h}:{m:02d}:{s:06.3f}" if h else f"{m}:{s:06.3f}"


def duong_dan_day_du(rel_path):
    """'Videos_L26_e/video/L26_V400.mp4' -> path tuyệt đối trong DATASET."""
    rel_path = rel_path.strip().strip('"').strip("'").replace("\\", "/")
    if os.path.isabs(rel_path) or rel_path.startswith("/kaggle"):
        return rel_path
    return os.path.join(DATASET, rel_path)


_cache_csv = {}

def lay_map_keyframes(video_id):
    """Đọc (và cache) DataFrame map-keyframes. None nếu không có."""
    if video_id not in _cache_csv:
        path = MAP_KEYFRAMES.get(video_id)
        _cache_csv[video_id] = pd.read_csv(path) if path else None
    return _cache_csv[video_id]


_cache_fps = {}

def lay_fps(video_path):
    """fps: ưu tiên map-keyframes (tức thì), chỉ mở video khi thật cần."""
    video_id = os.path.splitext(os.path.basename(video_path))[0]
    if video_id in _cache_fps:
        return _cache_fps[video_id]

    fps, nguon = None, None

    df = lay_map_keyframes(video_id)
    if df is not None and "fps" in df.columns:
        fps, nguon = float(df["fps"].iloc[0]), "map-keyframes"

    if fps is None and os.path.isfile(video_path):
        import cv2
        cap = cv2.VideoCapture(video_path)
        v = cap.get(cv2.CAP_PROP_FPS); cap.release()
        if v and v > 0:
            fps, nguon = float(v), "opencv"

    _cache_fps[video_id] = (fps, nguon)
    return _cache_fps[video_id]


## 3. Hàm tính chính

In [ ]:
def time_to_frame(rel_path, t, tim_keyframe=True):
    """Một video (path tương đối) + một mốc thời gian -> dict kết quả."""
    path = duong_dan_day_du(rel_path)
    video_id = os.path.splitext(os.path.basename(path))[0]
    giay = doc_thoi_gian(t)
    fps, nguon = lay_fps(path)

    kq = {
        "video": rel_path,
        "video_id": video_id,
        "thoi_gian": str(t),
        "so_giay": round(giay, 3),
        "fps": round(fps, 4) if fps else None,
        "frame_idx": int(round(giay * fps)) if fps else None,
        "nguon_fps": nguon,
    }

    # keyframe gần nhất (dùng DataFrame đã cache, không đọc lại file)
    if tim_keyframe and kq["frame_idx"] is not None:
        df = lay_map_keyframes(video_id)
        if df is not None and "frame_idx" in df.columns:
            i = (df["frame_idx"] - kq["frame_idx"]).abs().idxmin()
            row = df.loc[i]
            kq["kf_frame_idx"] = int(row["frame_idx"])
            kq["kf_n"] = int(row["n"]) if "n" in df.columns else int(i) + 1
            kq["kf_lech_giay"] = round((int(row["frame_idx"]) - kq["frame_idx"]) / fps, 3)
            kq["kf_anh"] = f"{kq['kf_n']:03d}.jpg"
    return kq


def frame_to_time(rel_path, frame_idx):
    """Chiều ngược: frame index -> mốc thời gian."""
    path = duong_dan_day_du(rel_path)
    fps, nguon = lay_fps(path)
    if not fps:
        return None
    return {"video": rel_path, "frame_idx": int(frame_idx), "fps": round(fps, 4),
            "so_giay": round(frame_idx / fps, 3),
            "thoi_gian": dinh_dang_thoi_gian(frame_idx / fps)}


## 4. ⭐ NHẬP NHIỀU VIDEO / NHIỀU THỜI GIAN Ở ĐÂY

Mỗi dòng: `đường_dẫn_tương_đối.mp4   thời_gian [, thời_gian, ...]`

- Đường dẫn chỉ cần phần sau `DATASET`: `Videos_L26_e/video/L26_V400.mp4`
- Dán cả path tuyệt đối `/kaggle/input/.../L26_V400.mp4` cũng được
- Thời gian: `mm:ss`, `hh:mm:ss`, hoặc số giây
- Dòng bắt đầu bằng `#` được bỏ qua

In [ ]:
INPUT = """
Videos_L26_e/video/L26_V400.mp4   5:01
Videos_L26_e/video/L26_V400.mp4   10:11, 12:30, 1:02:03
Videos_L26_e/video/L26_V401.mp4   0:45
# Videos_L01_a/video/L01_V001.mp4   2:15
"""


def doc_input(text):
    """Tách text nhiều dòng thành list (path, thời_gian)."""
    truy_van = []
    for dong in text.strip().splitlines():
        dong = dong.strip()
        if not dong or dong.startswith("#"):
            continue
        # tách path (kết thúc bằng .mp4) khỏi phần thời gian
        m = re.match(r"^(.*?\.mp4)\s*[,|\s]\s*(.+)$", dong, flags=re.IGNORECASE)
        if not m:
            print("Bỏ qua dòng không đúng định dạng:", dong)
            continue
        path, phan_tg = m.group(1).strip(), m.group(2)
        for t in [x for x in re.split(r"[\s,|]+", phan_tg) if x]:
            truy_van.append((path, t))
    return truy_van


truy_van = doc_input(INPUT)
print(f"{len(truy_van)} truy vấn:")
for v, t in truy_van:
    print("  ", v, "->", t)

## 5. Chạy — tính frame index & visualize ảnh luôn

Ảnh được **cắt trực tiếp từ file mp4** tại đúng `frame_idx` vừa tính
(`cap.set(CAP_PROP_POS_FRAMES, frame_idx)`), không lấy từ keyframe có sẵn.
Chạy cell dưới là ra cả bảng số liệu và lưới ảnh.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict

CAT_ANH = True   # False = chỉ lấy số liệu, không cắt ảnh
SO_COT  = 3      # số cột trong lưới ảnh


def doc_nhieu_frame(video_path, danh_sach_idx):
    """Mở video MỘT lần, seek lấy nhiều frame -> {frame_idx: ảnh RGB}."""
    ket_qua = {}
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("  [!] Không mở được video:", video_path)
        return ket_qua
    for idx in sorted(set(int(i) for i in danh_sach_idx)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            ket_qua[idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        else:
            print(f"  [!] Không đọc được frame {idx} trong {os.path.basename(video_path)}")
    cap.release()
    return ket_qua


def ve_luoi(danh_sach_kq, so_cot=3):
    """Hiển thị các ảnh đã cắt thành lưới."""
    co_anh = [r for r in danh_sach_kq if r.get("_anh") is not None]
    if not co_anh:
        print("Không có ảnh nào để hiển thị.")
        return co_anh

    n = len(co_anh)
    cot = min(so_cot, n)
    hang = (n + cot - 1) // cot
    fig, axes = plt.subplots(hang, cot, figsize=(5.8 * cot, 3.7 * hang))
    axes = [axes] if n == 1 else list(axes.flatten())

    for ax, r in zip(axes, co_anh):
        ax.imshow(r["_anh"])
        ax.set_title(f"{r['video_id']}  |  {r['thoi_gian']}  ->  frame {r['frame_idx']}\n"
                     f"fps {r['fps']}  |  {r.get('kich_thuoc', '')}", fontsize=9)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    return co_anh


# ---------- 1) Tính frame_idx ----------
ket_qua = [time_to_frame(v, t) for v, t in truy_van]

# ---------- 2) Cắt ảnh trực tiếp từ mp4 (mỗi video chỉ mở một lần) ----------
if CAT_ANH:
    can_cat = defaultdict(list)
    for r in ket_qua:
        if r["frame_idx"] is not None:
            can_cat[r["video"]].append(r["frame_idx"])

    anh_theo_video = {p: doc_nhieu_frame(duong_dan_day_du(p), ds)
                      for p, ds in can_cat.items()}

    for r in ket_qua:
        r["_anh"] = anh_theo_video.get(r["video"], {}).get(r["frame_idx"])
        if r["_anh"] is not None:
            r["kich_thuoc"] = f"{r['_anh'].shape[1]}x{r['_anh'].shape[0]}"

# ---------- 3) Bảng kết quả ----------
df_kq = pd.DataFrame([{k: v for k, v in r.items() if k != "_anh"} for r in ket_qua])
print(df_kq.to_string(index=False))

# ---------- 4) Visualize ----------
co_anh = ve_luoi(ket_qua, SO_COT) if CAT_ANH else []


### Lưu ảnh ra file

In [ ]:
OUT_DIR = "/kaggle/working/frames_tu_thoi_gian"
os.makedirs(OUT_DIR, exist_ok=True)

for r in co_anh:
    ten = f"{r['video_id']}_f{r['frame_idx']}.jpg"
    cv2.imwrite(os.path.join(OUT_DIR, ten), cv2.cvtColor(r["_anh"], cv2.COLOR_RGB2BGR))
    print("Đã lưu:", ten)


In [ ]:
# Dòng nào không tính được (không có fps trong map-keyframes và cũng không mở được video)
loi = df_kq[df_kq["fps"].isna()]
if len(loi):
    print("Không tính được:")
    print(loi[["video", "thoi_gian"]].to_string(index=False))
else:
    print("Tất cả đều tính được.")


In [ ]:
# Xuất CSV
OUT = "/kaggle/working/time_to_frame.csv"
df_kq.to_csv(OUT, index=False)
print("Đã lưu ->", OUT)

## 6. Chiều ngược lại: frame index → thời gian + xem ảnh frame

Nhập `đường_dẫn.mp4  frame_index [, frame_index, ...]`
→ trả về thời gian, fps, và **hiển thị ảnh** của frame đó (trích trực tiếp từ video).

In [ ]:
import cv2
import matplotlib.pyplot as plt


def doc_nhieu_frame(video_path, danh_sach_idx):
    """Mở video MỘT lần, seek lấy nhiều frame -> {frame_idx: ảnh RGB}."""
    ket_qua = {}
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return ket_qua
    for idx in sorted(set(int(i) for i in danh_sach_idx)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            ket_qua[idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    cap.release()
    return ket_qua


def frame_to_time_visual(rel_path, frame_idx, anh=None, hien_thi=True):
    """frame index -> thời gian + fps (+ ảnh frame nếu truyền vào hoặc tự đọc)."""
    path = duong_dan_day_du(rel_path)
    fps, nguon = lay_fps(path)
    if not fps:
        print("Không lấy được fps cho:", rel_path)
        return None

    giay = frame_idx / fps
    kq = {
        "video": rel_path,
        "video_id": os.path.splitext(os.path.basename(path))[0],
        "frame_idx": int(frame_idx),
        "fps": round(fps, 4),
        "nguon_fps": nguon,
        "so_giay": round(giay, 3),
        "thoi_gian": dinh_dang_thoi_gian(giay),
    }

    if anh is None and os.path.isfile(path):
        anh = doc_nhieu_frame(path, [frame_idx]).get(int(frame_idx))
    if anh is not None:
        kq["kich_thuoc"] = f"{anh.shape[1]}x{anh.shape[0]}"

    if hien_thi:
        if anh is None:
            print(f"[!] Không đọc được ảnh frame {frame_idx} của {rel_path}")
        else:
            plt.figure(figsize=(9, 5))
            plt.imshow(anh)
            plt.axis("off")
            plt.title(f"{kq['video_id']}  |  frame {kq['frame_idx']}  |  "
                      f"{kq['thoi_gian']}  |  fps {kq['fps']}", fontsize=11)
            plt.show()

    kq["_anh"] = anh
    return kq


### Nhập nhiều frame ở đây

In [ ]:
INPUT_FRAME = """
Videos_L26_e/video/L26_V400.mp4   7525, 15275
Videos_L26_e/video/L26_V401.mp4   1125
"""

# Gom theo video để mỗi file mp4 chỉ mở một lần (nhanh hơn nhiều)
from collections import defaultdict

nhom = defaultdict(list)
for path, f in doc_input(INPUT_FRAME):
    nhom[path].append(int(f))

ket_qua_frame = []
for path, danh_sach in nhom.items():
    anh_dict = doc_nhieu_frame(duong_dan_day_du(path), danh_sach)
    for idx in danh_sach:
        r = frame_to_time_visual(path, idx, anh=anh_dict.get(idx), hien_thi=True)
        if r:
            ket_qua_frame.append(r)

df_frame = pd.DataFrame([{k: v for k, v in r.items() if k != "_anh"} for r in ket_qua_frame])
df_frame


### Xem tất cả ảnh trong một lưới (grid)

In [ ]:
co_anh = [r for r in ket_qua_frame if r.get("_anh") is not None]

if co_anh:
    n = len(co_anh)
    cot = min(3, n)
    hang = (n + cot - 1) // cot
    fig, axes = plt.subplots(hang, cot, figsize=(6 * cot, 3.6 * hang))
    axes = [axes] if n == 1 else list(axes.flatten())

    for ax, r in zip(axes, co_anh):
        ax.imshow(r["_anh"])
        ax.set_title(f"{r['video_id']} | f{r['frame_idx']} | {r['thoi_gian']}", fontsize=9)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Không có ảnh nào để hiển thị.")

### Lưu ảnh các frame ra file

In [ ]:
OUT_DIR = "/kaggle/working/frames"
os.makedirs(OUT_DIR, exist_ok=True)

for r in co_anh:
    ten = f"{r['video_id']}_f{r['frame_idx']}.jpg"
    cv2.imwrite(os.path.join(OUT_DIR, ten),
                cv2.cvtColor(r["_anh"], cv2.COLOR_RGB2BGR))
    print("Đã lưu:", ten)